# Music by emotion

Classify the emotion of a music track using [`LaurenGurgiolo/Music_by_Emotion`](https://huggingface.co/LaurenGurgiolo/Music_by_Emotion), an Audio Spectrogram Transformer (AST) fine-tuned from `MIT/ast-finetuned-audioset-10-10-0.4593`.

The model expects mono audio at 16 kHz (it was trained on ~30-second clips).

Tracks compared below (everything in `My Drive/google_colab/music_examples`):
- `Andromida - Real.mp3`
- `Eminem - Lose Yourself.mp3`
- `Linkin Park - In The End.mp3`
- `Nero - Blame You.mp3`
- `Slipknot - Duality.mp3`
- `The Cardigans - My Favorite Game.mp3`

## 1. Install dependencies

In [5]:
!pip install -q transformers torch torchaudio librosa

## 2. Mount Google Drive

After mounting, `My Drive` is available at `/content/drive/MyDrive`.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

AUDIO_DIR = '/content/drive/MyDrive/google_colab/music_examples'
AUDIO_PATHS = [
    f'{AUDIO_DIR}/Andromida - Real.mp3',
    f'{AUDIO_DIR}/Eminem - Lose Yourself.mp3',
    f'{AUDIO_DIR}/Linkin Park - In The End.mp3',
    f'{AUDIO_DIR}/Nero - Blame You.mp3',
    f'{AUDIO_DIR}/Slipknot - Duality.mp3',
    f'{AUDIO_DIR}/The Cardigans - My Favorite Game.mp3',
]

Mounted at /content/drive


## 3. Load the model and processor

In [7]:
import torch
from transformers import AutoProcessor, AutoModelForAudioClassification

MODEL_ID = 'LaurenGurgiolo/Music_by_Emotion'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(MODEL_ID).to(device).eval()

# The published config only has generic LABEL_0..LABEL_10 names. These are
# the real classes, in the training dataset's ClassLabel order.
EMOTION_LABELS = [
    'Angry',      # 0
    'Contempt',   # 1
    'Disgust',    # 2
    'Fear',       # 3
    'Happy',      # 4
    'Neutral',    # 5
    'Sad',        # 6
    'Sleepy',     # 7
    'Surprised',  # 8
    'Bad',        # 9
    'Boring',     # 10
]

model.config.id2label = {i: name for i, name in enumerate(EMOTION_LABELS)}
model.config.label2id = {name: i for i, name in enumerate(EMOTION_LABELS)}

# 16 kHz for AST
TARGET_SR = processor.sampling_rate
print('Target sampling rate:', TARGET_SR)
print('Emotion labels:', model.config.id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/345M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Target sampling rate: 16000
Emotion labels: {0: 'Angry', 1: 'Contempt', 2: 'Disgust', 3: 'Fear', 4: 'Happy', 5: 'Neutral', 6: 'Sad', 7: 'Sleepy', 8: 'Surprised', 9: 'Bad', 10: 'Boring'}


## 4. Inference helper

Load a track (mono, resampled to the model's target rate), run the model, and
return the per-emotion probabilities.

In [8]:
import os
import librosa


def classify(path):
    """Return (probs_tensor, duration_seconds) for one audio file."""
    waveform, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    inputs = processor(waveform, sampling_rate=TARGET_SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu()
    return probs, len(waveform) / sr


# Classify every track once; the cells below just read `results`.
results = {}
for path in AUDIO_PATHS:
    name = os.path.splitext(os.path.basename(path))[0]
    probs, duration = classify(path)
    results[name] = probs
    print(f'{name}: {duration:.1f}s')

Andromida - Real: 218.1s
Eminem - Lose Yourself: 326.5s
Linkin Park - In The End: 219.0s
Nero - Blame You: 200.5s
Slipknot - Duality: 218.6s


/tmp/ipykernel_13962/1313586462.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  waveform, sr = librosa.load(path, sr=TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


The Cardigans - My Favorite Game: 216.3s


## 5. Top prediction per track

In [9]:
for name, probs in results.items():
    top_id = int(probs.argmax())
    print(f'{name}')
    print(f'  Predicted emotion: {model.config.id2label[top_id]} '
          f'({probs[top_id].item():.2%})\n')

Andromida - Real
  Predicted emotion: Bad (45.71%)

Eminem - Lose Yourself
  Predicted emotion: Sleepy (39.23%)

Linkin Park - In The End
  Predicted emotion: Disgust (40.92%)

Nero - Blame You
  Predicted emotion: Sad (55.21%)

Slipknot - Duality
  Predicted emotion: Bad (41.85%)

The Cardigans - My Favorite Game
  Predicted emotion: Bad (79.51%)



## 6. Side-by-side comparison

Full per-emotion probabilities for all six tracks, sorted by the first track's scores.

In [10]:
import pandas as pd

comparison = pd.DataFrame(
    {name: [probs[i].item() for i in range(len(EMOTION_LABELS))]
     for name, probs in results.items()},
    index=EMOTION_LABELS,
)
comparison.index.name = 'emotion'

# Sort by the first track so the table is easier to scan.
comparison = comparison.sort_values(by=comparison.columns[0], ascending=False)

(comparison * 100).round(2).style.format('{:.2f}%')

,Andromida - Real,Eminem - Lose Yourself,Linkin Park - In The End,Nero - Blame You,Slipknot - Duality,The Cardigans - My Favorite Game
emotion,,,,,,
Bad,45.71%,2.21%,6.67%,0.50%,41.85%,79.51%
Contempt,35.39%,0.09%,14.01%,11.57%,2.52%,0.38%
Surprised,8.06%,3.58%,2.72%,3.73%,0.11%,1.00%
Sleepy,3.04%,39.23%,2.27%,2.86%,0.72%,0.82%
Neutral,2.51%,0.54%,4.39%,0.12%,9.19%,0.36%
Sad,1.83%,28.22%,1.89%,55.21%,31.98%,0.25%
Fear,1.19%,0.06%,19.53%,0.99%,1.34%,0.56%
Angry,1.09%,21.86%,5.41%,0.72%,1.77%,1.28%
Happy,0.83%,1.68%,0.64%,0.01%,2.41%,10.44%
